# Lecture 06 - Neural Networks: figure generation

All lecture figures are generated here and saved as **PDF** into per-slide subfolders of `06_figures/` (e.g. `27_gradient_descent/`). Run top-to-bottom from this folder.

In [1]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa
warnings.filterwarnings("ignore")

# Notebook must be run from its own folder (99_Lecture).
try:
    BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE = os.getcwd()
CODE_ROOT = os.path.join(BASE, "..")
plt.style.use(os.path.join(CODE_ROOT, "configs", "visualisations.mplstyle"))
FIGROOT = os.path.join(BASE, "06_figures")

# palette (tableau-colorblind, shared across lectures)
BLUE="#006BA4"; ORANGE="#FF800E"; GRAY="#ABABAB"; DARK="#595959"; GREEN="#2C8A3B"; RED="#C0392B"
DIS_COL={"Earthquake":"#C0392B","Flood":"#2C6FA6","Storm":"#27AE60","Volcanic activity":"#E67E22"}
DIS_COLORS = DIS_COL  # alias

def load_geo(types=None):
    """Load EM-DAT natural-disaster records with valid coordinates."""
    fp = os.path.join(CODE_ROOT, "data", "1900_2021_DISASTERS.csv")
    df = pd.read_csv(fp, low_memory=False)
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
    df = df[df["Latitude"].between(-90, 90) & df["Longitude"].between(-180, 180)]
    types = types or ["Earthquake", "Flood", "Storm", "Volcanic activity"]
    return df[df["Disaster Type"].isin(types)], types

def sigmoid(z): return 1/(1+np.exp(-z))
def tanh(z): return np.tanh(z)
def relu(z): return np.maximum(0, z)

# Every figure is saved as PDF into 06_figures/<slide-folder>/ .
SLIDE_MAP = [
    ("mot_", "02_motivation"),
    ("activations_plain", "11_activation_functions"),
    ("activations", "34_activations"),
    ("softmax_disaster", "14_softmax"),
    ("cross_entropy", "17_cross_entropy"),
    ("nn_epoch_frame", "15_training_animation"),
    ("nn_epochs_4up", "15_training_animation"),
    ("gd1d_intro", "27_gradient_descent"),
    ("gd1d_step", "27_gradient_descent"),
    ("opt3d_", "37_momentum"),
    ("adam_vs_sgd", "38_adam"),
]

def _folder_for(name):
    for pre, fld in SLIDE_MAP:
        if name.startswith(pre):
            return fld
    return "_unsorted"

def save(fig, name):
    """Save a figure as PDF into its per-slide subfolder under 06_figures/."""
    if not name.endswith(".pdf"):
        name = name.rsplit(".", 1)[0] + ".pdf"
    folder = os.path.join(FIGROOT, _folder_for(name))
    os.makedirs(folder, exist_ok=True)
    fig.savefig(os.path.join(folder, name), bbox_inches="tight")
    plt.close(fig)
    print("saved", _folder_for(name) + "/" + name)

## Shared helpers

Math/geometry helpers used by several figures (1-D and 2-D/3-D loss landscapes).

In [2]:
def _L1d(w):
    return 0.03 * w ** 4 - 0.55 * w ** 2 + 0.25 * w + 4.0

def _dL1d(w):
    return 0.12 * w ** 3 - 1.10 * w + 0.25

_wg = np.linspace(-5, 5, 20001)

_Lg = _L1d(_wg)

_mins = _wg[1:-1][(_Lg[1:-1] < _Lg[:-2]) & (_Lg[1:-1] < _Lg[2:])]

GLOBAL_W = _mins[np.argmin(_L1d(_mins))]

LOCAL_W = _mins[np.argmax(_L1d(_mins))]

_SS_SPECS = [(0.010, "too small", RED, 0.1), (0.12, "well chosen", GREEN, -4.2), (1.35, "too large", RED, -4.2)]

def _ss_path(eta, w0, n):
    w = w0; pts = [w]
    for _ in range(n):
        w = w - eta * _dL1d(w); w = np.clip(w, -5.4, 5.4); pts.append(w)
    return np.array(pts)

def _S3(a, b):
    return a ** 3 - 3 * a * b ** 2

def _g3(p):
    return np.array([3 * p[0] ** 2 - 3 * p[1] ** 2, -6 * p[0] * p[1]])

_R = 1.25

_A3 = np.linspace(-_R, _R, 120)

_B3 = np.linspace(-_R, _R, 120)

_AA3, _BB3 = np.meshgrid(_A3, _B3)

_AA3, _BB3 = np.meshgrid(_A3, _B3)

_ZZ3 = _S3(_AA3, _BB3)

_ZOFF = 0.55

def _z_on(path):
    return np.array([_S3(x, y) for x, y in path]) + _ZOFF

def _surf3d(ax):
    ax.plot_surface(_AA3, _BB3, _ZZ3, cmap="viridis", alpha=1.0, linewidth=0,
                    antialiased=True, rcount=120, ccount=120)
    ax.set_xlabel("$\\theta_1$"); ax.set_ylabel("$\\theta_2$"); ax.set_zlabel(r"$R(\theta)$")
    ax.set_zlim(_ZZ3.min() - 0.2, _ZZ3.max() + 0.6)
    ax.view_init(elev=38, azim=-52)

def _bowl(A, B):
    return 0.5 * A ** 2 + 1.3 * B ** 2

def _grad(a, b):
    return np.array([a, 2.6 * b])

## Slide 2 - Motivation: nonlinear decision boundaries

In [3]:
def motivation_panels():
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    df, _ = load_geo(["Earthquake", "Flood"])
    df = df.sample(min(900, len(df)), random_state=0)
    X = df[["Longitude", "Latitude"]].values.astype(float)
    y = (df["Disaster Type"].values == "Flood").astype(int)
    xx, yy = np.meshgrid(np.linspace(-180, 180, 400), np.linspace(-90, 90, 220))
    grid = np.c_[xx.ravel(), yy.ravel()]
    cmap = ListedColormap(["#E8A9A0", "#A7C3DE"]); A = 0.6

    def sig_feats(P):
        s = 1 / (1 + np.exp(-P / 45.0))
        return np.column_stack([P, s, s ** 2, P[:, 0] * P[:, 1] / 100.0])

    def panel(name, Z, title):
        fig, ax = plt.subplots(figsize=(4.0, 3.5))
        if Z is not None:
            ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], cmap=cmap, alpha=A)
        for lab, col, nm in [(0, DIS_COL["Earthquake"], "Earthquake"), (1, DIS_COL["Flood"], "Flood")]:
            m = y == lab
            ax.scatter(X[m, 0], X[m, 1], s=10, color=col, alpha=0.9, label=nm, edgecolors="white", linewidth=0.15)
        ax.set_xlim(-180, 180); ax.set_ylim(-90, 90); ax.grid(False)
        ax.set_xlabel("Longitude", fontsize=9); ax.set_ylabel("Latitude", fontsize=9)
        ax.set_title(title, fontsize=12)
        if name == "p0":
            ax.legend(loc="lower left", fontsize=8, markerscale=1.4)
        save(fig, f"mot_{name}.pdf")

    panel("p0", None, "Raw data")
    lin = LinearRegression().fit(X, y)
    panel("p1", (lin.predict(grid) > 0.5).astype(int).reshape(xx.shape), "Linear regression")
    log = LogisticRegression(max_iter=3000).fit(sig_feats(X), y)
    panel("p2", log.predict(sig_feats(grid)).reshape(xx.shape), "Logistic regression")
    tree = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X, y)
    panel("p3", tree.predict(grid).reshape(xx.shape), "Decision tree (depth 5)")

motivation_panels()

saved 02_motivation/mot_p0.pdf
saved 02_motivation/mot_p1.pdf


saved 02_motivation/mot_p2.pdf
saved 02_motivation/mot_p3.pdf


## Slide 11 - Activation functions (plain)

In [4]:
def fig_activations_plain():
    """Slide 11: activation curves g(z) without derivatives."""
    z = np.linspace(-5, 5, 400)
    funcs = [("Sigmoid", sigmoid(z)), ("Hyperbolic tangent", tanh(z)), ("ReLU", relu(z))]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
    for ax, (name, g) in zip(axes, funcs):
        ax.plot(z, g, color=BLUE, lw=1.8, label="$g(z)$")
        ax.set_title(name, fontsize=11); ax.set_xlabel("$z$")
        ax.axhline(0, color=GRAY, lw=0.5); ax.axvline(0, color=GRAY, lw=0.5)
    save(fig, "activations_plain.pdf")

fig_activations_plain()

saved 11_activation_functions/activations_plain.pdf


## Slide 14 - Softmax over the disaster classes

In [5]:
def softmax_disaster():
    classes = ["Earthquake", "Flood", "Storm", "Volcanic"]
    cols = [DIS_COL["Earthquake"], DIS_COL["Flood"], DIS_COL["Storm"], DIS_COL["Volcanic activity"]]
    logits = np.array([3.1, 0.7, -0.4, 1.2])
    p = np.exp(logits) / np.exp(logits).sum()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    axes[0].bar(classes, logits, color=GRAY)
    axes[0].axhline(0, color=DARK, lw=0.6)
    axes[0].set_title(r"raw scores (logits)  $Z_m$", fontsize=11)
    axes[0].set_ylabel("score")
    b = axes[1].bar(classes, p, color=cols)
    for i, v in enumerate(p):
        axes[1].text(i, v + 0.015, f"{v:.2f}", ha="center", fontsize=10, weight="bold")
    axes[1].set_ylim(0, 1)
    axes[1].set_title(r"softmax  $f_m(X)$   (sum = %.2f)" % p.sum(), fontsize=11)
    axes[1].set_ylabel("probability")
    # cumulative sum annotation
    axes[1].axhline(1.0, color=DARK, ls=":", lw=1)
    axes[1].text(3.4, 1.0, "1.0", va="bottom", ha="right", fontsize=9, color=DARK)
    for ax in axes:
        ax.tick_params(axis="x", labelsize=9)
    save(fig, "softmax_disaster.pdf")

softmax_disaster()

saved 14_softmax/softmax_disaster.pdf


## Slides 15 & 41 - Training the classifier over epochs

In [6]:
def nn_over_epochs():
    from sklearn.neural_network import MLPClassifier
    from sklearn.preprocessing import StandardScaler
    df, types = load_geo()
    X = df[["Longitude", "Latitude"]].values.astype(float)
    y = df["Disaster Type"].astype(str).values
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    clf = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                        max_iter=1, warm_start=True, random_state=0, learning_rate_init=0.02)
    xx, yy = np.meshgrid(np.linspace(-180, 180, 420), np.linspace(-90, 90, 240))
    grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
    classes = None
    snaps = [1, 3, 8, 20, 60, 150]
    losses = []
    snap_data = {}
    ep = 0
    maxep = max(snaps)
    while ep < maxep:
        clf.fit(Xs, y); ep += 1
        losses.append(clf.loss_)
        if ep in snaps:
            classes = list(clf.classes_)
            code = {c: i for i, c in enumerate(classes)}
            Z = np.vectorize(code.get)(clf.predict(grid)).reshape(xx.shape)
            snap_data[ep] = (Z, list(losses))
    cmap = ListedColormap([DIS_COL[c] for c in classes])

    def draw_map(ax, Z):
        ax.contourf(xx, yy, Z, levels=np.arange(-0.5, len(classes) + 0.5), cmap=cmap, alpha=0.25, zorder=0)
        for c in classes:
            s = df[df["Disaster Type"] == c]
            ax.scatter(s["Longitude"], s["Latitude"], s=5, alpha=0.6, color=DIS_COL[c],
                       edgecolors="white", linewidth=0.15, zorder=2)
        ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
        ax.set_xlabel("Longitude", fontsize=9); ax.set_ylabel("Latitude", fontsize=9)

    # 4-up static (epochs 1,8,20,150)
    fig, axes = plt.subplots(2, 2, figsize=(11, 6))
    for ax, e in zip(axes.ravel(), [1, 8, 20, 150]):
        draw_map(ax, snap_data[e][0]); ax.set_title(f"epoch {e}", fontsize=11)
    fig.tight_layout()
    save(fig, "nn_epochs_4up.pdf")

    # animation frames: left loss curve so far, right map at that epoch
    for i, e in enumerate(snaps):
        Z, lh = snap_data[e]
        fig, (axl, axr) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
        axl.plot(np.arange(1, len(lh) + 1), lh, color=BLUE, lw=2)
        axl.scatter([len(lh)], [lh[-1]], color=ORANGE, s=30, zorder=5)
        axl.set_xlim(1, maxep); axl.set_ylim(0.6, max(1.25, max(losses[:3]) + 0.05))
        axl.set_xlabel("epoch"); axl.set_ylabel("training loss (cross-entropy)")
        axl.set_title(f"epoch {e}", fontsize=11)
        draw_map(axr, Z)
        save(fig, f"nn_epoch_frame_{i}.pdf")

nn_over_epochs()

saved 15_training_animation/nn_epochs_4up.pdf
saved 15_training_animation/nn_epoch_frame_0.pdf


saved 15_training_animation/nn_epoch_frame_1.pdf
saved 15_training_animation/nn_epoch_frame_2.pdf


saved 15_training_animation/nn_epoch_frame_3.pdf


saved 15_training_animation/nn_epoch_frame_4.pdf
saved 15_training_animation/nn_epoch_frame_5.pdf


## Slide 17 - Cross-entropy loss

In [7]:
def fig_cross_entropy():
    p = np.linspace(1e-3, 1, 300)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(p, -np.log(p), color=BLUE, lw=2)
    ax.fill_between(p, -np.log(p), color=BLUE, alpha=0.08)
    ax.set_xlabel("predicted probability assigned to the true class $\\hat{p}_y$")
    ax.set_ylabel("loss  $-\\log \\hat{p}_y$")
    ax.set_ylim(0, 5)
    ax.set_xlim(0, 1)
    ax.annotate("confident & correct\n$\\rightarrow$ loss $\\approx 0$", xy=(0.96, 0.04),
                xytext=(0.55, 0.9), fontsize=9,
                arrowprops=dict(arrowstyle="-|>", color=DARK, lw=1.2))
    ax.annotate("confident & wrong\n$\\rightarrow$ loss $\\rightarrow \\infty$", xy=(0.03, -np.log(0.03)),
                xytext=(0.18, 3.4), fontsize=9,
                arrowprops=dict(arrowstyle="-|>", color=DARK, lw=1.2))
    ax.set_title("Cross-entropy punishes confident, wrong predictions", fontsize=11)
    save(fig, "cross_entropy.pdf")

fig_cross_entropy()

saved 17_cross_entropy/cross_entropy.pdf


## Slide 27 - Gradient descent (1-D build)

In [8]:
def gd_1d_intro():
    """Build-up frames that precede the gradient-descent steps.

    Three frames share the exact axes, limits, title height and labels of
    ``gd_1d_build`` so they overlay pixel-aligned when stacked on one slide:
    (1) empty axes to introduce the axis, (2) the loss curve, (3) the minima.
    """
    ws = np.linspace(-4.6, 4.6, 500)

    def _base():
        fig, ax = plt.subplots(figsize=(8.6, 5.0))
        ax.set_xlabel(r"parameter  $\theta$"); ax.set_ylabel(r"$R(\theta)$")
        ax.set_title("gradient-descent", fontsize=11)   # keep title height constant
        ax.set_xlim(-4.6, 4.6); ax.set_ylim(0, 8)
        return fig, ax

    # (1) empty axes
    fig, ax = _base()
    save(fig, "gd1d_intro_axes.pdf")

    # (2) loss curve only
    fig, ax = _base()
    ax.plot(ws, _L1d(ws), color=BLUE, lw=2.4, label=r"$R(\theta)$")
    ax.legend(loc="upper center", fontsize=9)
    save(fig, "gd1d_intro_curve.pdf")

    # (3) curve + minima
    fig, ax = _base()
    ax.plot(ws, _L1d(ws), color=BLUE, lw=2.4, label=r"$R(\theta)$")
    ax.scatter([GLOBAL_W, LOCAL_W], [_L1d(GLOBAL_W), _L1d(LOCAL_W)],
               color=GREEN, s=45, zorder=5, label="minima")
    ax.legend(loc="upper center", fontsize=9)
    save(fig, "gd1d_intro_minima.pdf")

def gd_1d_build():
    ws = np.linspace(-4.6, 4.6, 500)
    w, eta = -4.2, 0.12
    path = [w]
    for _ in range(7):
        w = w - eta * _dL1d(w); path.append(w)
    for k in range(len(path)):
        fig, ax = plt.subplots(figsize=(8.6, 5.0))
        ax.plot(ws, _L1d(ws), color=BLUE, lw=2.4, label=r"$R(\theta)$")
        ax.scatter([GLOBAL_W, LOCAL_W], [_L1d(GLOBAL_W), _L1d(LOCAL_W)],
                   color=GREEN, s=45, zorder=5, label="minima")
        wk = path[k]; slope = _dL1d(wk); Lk = _L1d(wk)
        tx = np.linspace(wk - 1.3, wk + 1.3, 10)
        ax.plot(tx, Lk + slope * (tx - wk), color=ORANGE, lw=2, ls="--", label=r"$\nabla R$")
        ax.scatter([wk], [Lk], color=RED, s=75, zorder=6)
        if k < len(path) - 1:
            wn = path[k + 1]
            ax.annotate("", xy=(wn, _L1d(wk)), xytext=(wk, _L1d(wk)),
                        arrowprops=dict(arrowstyle="-|>", color=DARK, lw=2))
        ax.set_xlabel(r"parameter  $\theta$"); ax.set_ylabel(r"$R(\theta)$")
        ax.set_title(f"gradient-descent step {k}", fontsize=11)
        ax.legend(loc="upper center", fontsize=9)
        ax.set_xlim(-4.6, 4.6); ax.set_ylim(0, 8)   # FIXED limits so the plot never jumps
        save(fig, f"gd1d_step_{k:02d}.pdf")

gd_1d_intro()
gd_1d_build()

saved 27_gradient_descent/gd1d_intro_axes.pdf
saved 27_gradient_descent/gd1d_intro_curve.pdf


saved 27_gradient_descent/gd1d_intro_minima.pdf
saved 27_gradient_descent/gd1d_step_00.pdf


saved 27_gradient_descent/gd1d_step_01.pdf
saved 27_gradient_descent/gd1d_step_02.pdf


saved 27_gradient_descent/gd1d_step_03.pdf
saved 27_gradient_descent/gd1d_step_04.pdf


saved 27_gradient_descent/gd1d_step_05.pdf
saved 27_gradient_descent/gd1d_step_06.pdf


saved 27_gradient_descent/gd1d_step_07.pdf


## Slide 34 - Activation functions with derivatives

In [9]:
def fig_activations():
    z = np.linspace(-5, 5, 400)
    funcs = [("Sigmoid", sigmoid(z), sigmoid(z) * (1 - sigmoid(z))),
             ("Hyperbolic tangent", tanh(z), 1 - tanh(z) ** 2),
             ("ReLU", relu(z), (z >= 0).astype(float))]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
    for ax, (name, g, gp) in zip(axes, funcs):
        ax.plot(z, g, color=BLUE, lw=1.8, label="$g(z)$")
        ax.plot(z, gp, color=ORANGE, lw=1.6, ls="--", label="$g'(z)$")
        ax.set_title(name, fontsize=11)
        ax.set_xlabel("$z$")
        ax.axhline(0, color=GRAY, lw=0.5)
        ax.axvline(0, color=GRAY, lw=0.5)
        ax.legend(loc="upper left", fontsize=9)
    save(fig, "activations.pdf")

fig_activations()

saved 34_activations/activations.pdf


## Slide 37 - Momentum (optimizer paths in 3-D)

In [10]:
def opt3d_anim():
    # three methods starting near the saddle point (slide 30)
    start = np.array([0.06, 0.05]); eta = 0.05
    p = start.copy(); batch = [p.copy()]
    for _ in range(24):
        p = p - eta * _g3(p); p = np.clip(p, -_R, _R); batch.append(p.copy())
    rng = np.random.default_rng(3); p = start.copy(); sgd = [p.copy()]
    for _ in range(24):
        p = p - eta * (_g3(p) + rng.normal(0, 0.5, 2)); p = np.clip(p, -_R, _R); sgd.append(p.copy())
    p = start.copy(); v = np.zeros(2); mom = [p.copy()]
    for _ in range(24):
        v = 0.85 * v - eta * _g3(p); p = np.clip(p + v, -_R, _R); mom.append(p.copy())
    batch, sgd, mom = map(np.array, (batch, sgd, mom))
    frames = [1, 4, 8, 13, 18, 25]
    for fi, k in enumerate(frames):
        fig = plt.figure(figsize=(9, 6)); ax = fig.add_subplot(111, projection="3d")
        _surf3d(ax)
        for pathv, col, name in [(sgd, "#FF7A1A", "basic SGD"), (batch, "#111111", "batch gradient descent"),
                                 (mom, "#E23B2E", "with momentum")]:
            pp = pathv[:k]
            ax.plot(pp[:, 0], pp[:, 1], _z_on(pp), "-o", color=col, lw=2.4, ms=3.5, zorder=10, label=name)
        ax.legend(loc="upper left", fontsize=9)
        ax.set_title("optimizers near the saddle point ($Z=x^3-3xy^2$)", fontsize=10.5)
        save(fig, f"opt3d_{fi:02d}.pdf")

opt3d_anim()

saved 37_momentum/opt3d_00.pdf


saved 37_momentum/opt3d_01.pdf


saved 37_momentum/opt3d_02.pdf


saved 37_momentum/opt3d_03.pdf


saved 37_momentum/opt3d_04.pdf


saved 37_momentum/opt3d_05.pdf


## Slide 38 - Adam vs SGD

In [11]:
def adam_vs_sgd():
    a = np.linspace(-3, 3, 300); b = np.linspace(-2.2, 2.2, 240)
    A, B = np.meshgrid(a, b); Z = _bowl(A, B)
    rng = np.random.default_rng(1)
    p = np.array([-2.6, 1.9]); sgd = [p.copy()]
    for _ in range(30):
        p = p - 0.16 * (_grad(*p) + rng.normal(0, 1.1, 2)); sgd.append(p.copy())
    sgd = np.array(sgd)
    p = np.array([-2.6, 1.9]); m = np.zeros(2); v = np.zeros(2); adam = [p.copy()]
    b1, b2, eta, eps = 0.9, 0.999, 0.35, 1e-8
    for t in range(1, 26):
        gg = _grad(*p) + rng.normal(0, 0.5, 2)
        m = b1 * m + (1 - b1) * gg; v = b2 * v + (1 - b2) * gg * gg
        mh = m / (1 - b1 ** t); vh = v / (1 - b2 ** t)
        p = p - eta * mh / (np.sqrt(vh) + eps); adam.append(p.copy())
    adam = np.array(adam)
    fig, ax = plt.subplots(figsize=(9, 5.6))
    ax.contour(A, B, Z, levels=16, colors=GRAY, linewidths=0.4)
    ax.plot(sgd[:, 0], sgd[:, 1], "-o", color=ORANGE, lw=1.2, ms=3, alpha=0.8, label="SGD (noisy, no momentum)")
    ax.plot(adam[:, 0], adam[:, 1], "-o", color=BLUE, lw=2, ms=4, label="Adam (momentum + adaptive)")
    ax.scatter([0], [0], marker="*", color=RED, s=200, zorder=6, label="minimum")
    ax.set_xlabel("$\\theta_1$"); ax.set_ylabel("$\\theta_2$"); ax.legend(loc="upper right", fontsize=9)
    ax.set_aspect("equal")
    save(fig, "adam_vs_sgd.pdf")

adam_vs_sgd()

saved 38_adam/adam_vs_sgd.pdf


## Slides 35-38 - Shared 3-D loss surface (rotation + descent frames)

Plucked-string surface; rotation and gradient-descent frames saved under `06_figures/loss_surface/`.

In [12]:
OUT = os.path.join(FIGROOT, "loss_surface")
ROT = os.path.join(OUT, "rotation")
DESC = os.path.join(OUT, "descent")

CMAP = "turbo"
ELEV, AZIM = 30, -120        # default camera (5th angle)
DESC_AZIM = AZIM + 6 * 14     # descent view = 15th rotation frame (azim = -36)
ZRATIO = 0.3                 # box aspect -> gentle slopes
N_ROT = 24                   # rotation frames per revolution
START = (-0.6, 1.8)          # descent start (reaches global min)
LR = 0.8
FIGSIZE = (6.2, 5.2)

# ---- surface definition -------------------------------------------------
_xx = np.linspace(0, np.pi, 4000)
_A = np.array([(2 / np.pi) * np.trapezoid(_xx**3 * (np.pi - _xx) * np.sin(j * _xx), _xx)
               for j in range(1, 4)])
_J = np.arange(1, 4)
_KX, _KY = np.pi / 6.0, np.pi / 3.0
_g = np.linspace(-3, 3, 240)
_X, _Y = np.meshgrid(_g, _g)
_Zraw = sum(_A[k] * np.cos(_J[k] * ((_Y + 3) * _KY)) * np.sin(_J[k] * ((_X + 3) * _KX))
            for k in range(3))
_C = 1.0 / np.max(np.abs(_Zraw))
Z = _Zraw * _C


def loss(x, y):
    """Loss surface value R(x, y)."""
    return _C * np.sum(_A * np.cos(_J * ((y + 3) * _KY)) * np.sin(_J * ((x + 3) * _KX)))


def grad(x, y):
    """Analytic gradient of the loss surface at (x, y)."""
    dfx = _C * np.sum(_A * np.cos(_J * ((y + 3) * _KY)) * np.cos(_J * ((x + 3) * _KX)) * _J * _KX)
    dfy = _C * np.sum(_A * (-np.sin(_J * ((y + 3) * _KY))) * _J * _KY * np.sin(_J * ((x + 3) * _KX)))
    return np.array([dfx, dfy])


_mi = np.unravel_index(np.argmin(Z), Z.shape)
GMIN = (_X[_mi], _Y[_mi], Z[_mi])


def descent_path(start=START, lr=LR, n_steps=11):
    """Return the gradient-descent trajectory as an (n+1, 2) array."""
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for _ in range(n_steps):
        p = np.clip(p - lr * grad(*p), -3, 3)
        path.append(p.copy())
    return np.array(path)


# ---- drawing ------------------------------------------------------------
def _base_axes(fig, azim=AZIM):
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(_X, _Y, Z, cmap=CMAP, rcount=44, ccount=44,
                    linewidth=0.1, edgecolors=(0, 0, 0, 0.18), antialiased=True, alpha=0.95)
    ax.set_box_aspect((1, 1, ZRATIO))
    ax.set_zlim(-1, 1)
    ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$"); ax.set_zlabel(r"$R(\theta)$")
    for _pax in (ax.xaxis, ax.yaxis, ax.zaxis):      # clean white panes (no grey box)
        _pax.set_pane_color((1, 1, 1, 1))
        _pax.pane.set_edgecolor((0.85, 0.85, 0.85, 1))
    ax.view_init(elev=ELEV, azim=azim)
    return ax


def static_surface():
    fig = plt.figure(figsize=FIGSIZE)
    ax = _base_axes(fig)
    ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=60, zorder=7)
    fig.savefig(os.path.join(OUT, "loss_surface.pdf"), bbox_inches="tight")
    plt.close(fig)
    print("saved loss_surface/loss_surface.pdf")


def rotation_frames(n=N_ROT):
    """One PDF per rotation step; identical fixed canvas for clean overlay."""
    for i in range(n):
        az = AZIM + 360 * i / n
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=az)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=45, zorder=7)
        fig.savefig(os.path.join(ROT, f"rotation_{i:02d}.pdf"))
        plt.close(fig)
    print(f"saved rotation/rotation_00..{n - 1:02d}.pdf")


def descent_frames():
    """One PDF per gradient-descent step; identical fixed canvas for clean overlay."""
    path = descent_path()
    zz = np.array([loss(x, y) + 0.03 for x, y in path])
    for k in range(len(path)):
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=DESC_AZIM)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=55, zorder=7,
                   label="global minimum")
        ax.plot(path[:k + 1, 0], path[:k + 1, 1], zz[:k + 1], "-o",
                color="k", ms=4, lw=1.8, zorder=8)
        ax.scatter([path[0, 0]], [path[0, 1]], [zz[0]], color="magenta", s=55,
                   zorder=9, label=f"start {START}")
        ax.scatter([path[k, 0]], [path[k, 1]], [zz[k]], color="red", s=60, zorder=10)
        ax.set_title(rf"$\rho={LR}$    step ${k}$", fontsize=11)
        ax.legend(loc="upper right", fontsize=8)
        fig.savefig(os.path.join(DESC, f"descent_{k:02d}.pdf"))
        plt.close(fig)
    print(f"saved descent/descent_00..{len(path) - 1:02d}.pdf")


for _d in (OUT, ROT, DESC):
    os.makedirs(_d, exist_ok=True)
static_surface()
rotation_frames()
descent_frames()
print("loss_surface done; global min ~ (%.2f, %.2f)" % (GMIN[0], GMIN[1]))

saved loss_surface/loss_surface.pdf


saved rotation/rotation_00..23.pdf


saved descent/descent_00..11.pdf
loss_surface done; global min ~ (-1.34, 0.01)


## Slides 36-38 - learning rate regimes (2-D)

Gradient descent on the shared loss surface from the same start (2.0, 2.5) for three step sizes, one slide each: too large, too small, well chosen. PDF frames in `36_lr_too_large/`, `37_lr_too_small/`, `38_lr_optimal/`.

In [13]:
# Slides 36-38 - learning rate on the 2-D surface, one slide per regime.
# too large -> overshoots (slide 36); too small -> stalls in a valley (slide 37);
# well chosen -> reaches the global minimum (slide 38).
LR_REGIMES = {
    "too_high":    (2.5,  "36_lr_too_large"),
    "too_small":   (0.05, "37_lr_too_small"),
    "well_chosen": (1.2,  "38_lr_optimal"),
}
START_LR = (2.0, 2.5)   # sits on a small valley -> path changes direction

def lr_descent_frames(regime, lr, folder, n_steps=30):
    out = os.path.join(FIGROOT, folder)
    os.makedirs(out, exist_ok=True)
    path = descent_path(start=START_LR, lr=lr, n_steps=n_steps)
    zz = np.array([loss(x, y) + 0.03 for x, y in path])
    for k in range(len(path)):
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=DESC_AZIM)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=55,
                   zorder=7, label="minimum")
        ax.plot(path[:k + 1, 0], path[:k + 1, 1], zz[:k + 1], "-o",
                color="k", ms=4, lw=1.8, zorder=8)
        ax.scatter([path[0, 0]], [path[0, 1]], [zz[0]], color="magenta", s=55,
                   zorder=9, label=f"start {START_LR}")
        ax.scatter([path[k, 0]], [path[k, 1]], [zz[k]], color="red", s=60, zorder=10)
        ax.set_title(rf"$\rho={lr}$    step ${k}$", fontsize=11)
        ax.legend(loc="upper right", fontsize=8)
        if k % 5 == 0 or k < 5:                      # dense early, then every 5th step
            fig.savefig(os.path.join(out, f"step_{k:02d}.pdf"), bbox_inches="tight")
        plt.close(fig)
    print(f"saved {folder}: sampled frames of {len(path)}")

for _regime, (_lr, _folder) in LR_REGIMES.items():
    lr_descent_frames(_regime, _lr, _folder)

saved 36_lr_too_large: sampled frames of 31


saved 37_lr_too_small: sampled frames of 31


saved 38_lr_optimal: sampled frames of 31


## Slide 39 - SGD, mini-batch, full batch (gradient noise)

At one point on the loss surface: the true (full-batch) gradient, a cloud of noisy per-sample gradients, and the batch-mean estimate. As the batch size B grows the mean rotates onto the true gradient (error shrinks like 1/sqrt(B)). Frames in `39_sgd_minibatch/`.

In [14]:
# Slide 39 - SGD / mini-batch / full batch on the same loss surface.
# The gradient from one sample is a NOISY version of the true (full-batch) gradient.
# Averaging over B samples shrinks the noise (error ~ 1/sqrt(B)), so the batch
# estimate rotates onto the true gradient as B grows.
from matplotlib.lines import Line2D

def sgd_gradient_noise_frames(theta0=(0.8, 1.2), N=256, seed=1):
    out = os.path.join(FIGROOT, "39_sgd_minibatch")
    os.makedirs(out, exist_ok=True)
    t0 = np.array(theta0, float)
    g_true = grad(*t0)                                  # true (full-batch) gradient
    rng = np.random.default_rng(seed)
    sigma = 1.3 * np.linalg.norm(g_true)                # per-sample noise level
    samples = g_true + rng.normal(0, sigma, size=(N, 2))  # noisy per-sample gradients
    scale = 1.1 / np.linalg.norm(g_true)                # arrow length scaling

    def arrow(ax, vec, color, lw, alpha=1.0, ls="-", z=5):
        # draw the (descent) step direction -vec as an arrow from t0
        ax.annotate("", xy=t0 - scale * vec, xytext=t0,
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                    alpha=alpha, linestyle=ls), zorder=z)

    for B in [1, 2, 4, 8, 16, 64, N]:
        fig, ax = plt.subplots(figsize=(6.0, 5.6))
        ax.contour(_X, _Y, Z, levels=16, colors="#B9C4CE", linewidths=0.6, zorder=0)
        for s in samples[:B]:                           # per-sample gradients (noisy)
            arrow(ax, s, DARK, 1.0, alpha=0.50, z=2)
        arrow(ax, g_true, BLUE, 2.4, ls="--", z=6)      # true gradient
        arrow(ax, samples[:B].mean(0),  ORANGE, 2.8, z=7)
        ax.scatter(*t0, color="k", s=30, zorder=8)
        name = (f"Gradient decent, {B} sample(s)")
        ax.set_title(name, fontsize=11)
        ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
        ax.legend(handles=[
            Line2D([0], [0], color=BLUE, ls="--", lw=2, label="true gradient"),
            Line2D([0], [0], color=ORANGE, lw=2.5, label="batch estimate"),
            Line2D([0], [0], color=DARK, lw=2,alpha=0.50, label="per-sample gradients")],
            loc="upper right", fontsize=8)
        fig.savefig(os.path.join(out, f"batch_{B:03d}.pdf"), bbox_inches="tight")
        plt.close(fig)
    print("saved 39_sgd_minibatch: batch frames B =", [1, 2, 4, 8, 16, 64, N])

sgd_gradient_noise_frames()

saved 39_sgd_minibatch: batch frames B = [1, 2, 4, 8, 16, 64, 256]


## Slide 40 - momentum (2-D)

Same loss surface, same start. Plain GD stalls on the local shoulder; momentum accumulates velocity ($v \leftarrow \gamma v - \rho\,\nabla R$; $\theta \leftarrow \theta + v$) and rolls through into the global minimum. Frames in `40_momentum/`.

In [15]:
# Slide 40 - momentum on the same loss surface.
# Plain GD (rho=0.35) stalls on the local shoulder near (1, 1); momentum
# (v <- gamma*v - rho*grad(theta); theta <- theta + v) carries velocity from
# previous steps and rolls through it into the global minimum.
def momentum_frames(start=(2.0, 2.5), rho=0.35, gamma=0.8, n_steps=30):
    out = os.path.join(FIGROOT, "40_momentum")
    os.makedirs(out, exist_ok=True)
    # plain gradient descent
    p = np.array([start[0] + 0.1, start[1]], float); plain = [p.copy()]
    for _ in range(n_steps):
        p = np.clip(p - rho * grad(*p), -3, 3); plain.append(p.copy())
    plain = np.array(plain)
    # momentum
    q = np.array(start, float); v = np.zeros(2); mom = [q.copy()]
    for _ in range(n_steps):
        v = gamma * v - rho * grad(*q); q = np.clip(q + v, -3, 3); mom.append(q.copy())
    mom = np.array(mom)
    zp = np.array([loss(x, y) + 0.03 for x, y in plain])
    zm = np.array([loss(x, y) + 0.03 for x, y in mom])
    for k in range(n_steps + 1):
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=DESC_AZIM)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=55,
                   zorder=7, label="minimum")
        ax.plot(plain[:k + 1, 0], plain[:k + 1, 1], zp[:k + 1], "-o",
                color="black", ms=3, lw=1.6, zorder=8, label="plain GD")
        ax.plot(mom[:k + 1, 0], mom[:k + 1, 1], zm[:k + 1], "-o",
                color=ORANGE, ms=3.5, lw=2.0, zorder=9, label="momentum")
        ax.scatter([start[0]], [start[1]], [zp[0]], color="magenta", s=50, zorder=10)
        ax.set_title(rf"momentum vs plain GD    step ${k}$", fontsize=11)
        ax.legend(loc="upper right", fontsize=8)
        if k % 5 == 0 or k < 5:
            fig.savefig(os.path.join(out, f"step_{k:02d}.pdf"), bbox_inches="tight")
        plt.close(fig)
    print("saved 40_momentum: sampled frames of", n_steps + 1)

momentum_frames()

saved 40_momentum: sampled frames of 31


## Slide 41 - Adam (2-D)

Same loss surface and same $\rho$ as plain GD. Adam keeps a momentum term and a per-parameter adaptive scale, so it rolls through the shoulder and reaches the global minimum where plain GD stalls. Frames in `41_adam/`.

In [16]:
# Slide 41 - Adam on the same loss surface.
# Adam = momentum (1st moment m) + per-parameter adaptive step (2nd moment v).
# Same rho as plain GD: plain stalls on the local shoulder, Adam reaches the minimum.
def adam_frames(start=(2.0, 2.5), rho=0.3, b1=0.9, b2=0.999, eps=1e-8, n_steps=40):
    out = os.path.join(FIGROOT, "41_adam")
    os.makedirs(out, exist_ok=True)
    # plain GD (started 0.01 off so both curves are visible from step 0)
    p = np.array([start[0] + 0.01, start[1]], float); plain = [p.copy()]
    for _ in range(n_steps):
        p = np.clip(p - rho * grad(*p), -3, 3); plain.append(p.copy())
    plain = np.array(plain)
    # Adam
    q = np.array(start, float); m = np.zeros(2); v = np.zeros(2); adm = [q.copy()]
    for t in range(1, n_steps + 1):
        gg = grad(*q)
        m = b1 * m + (1 - b1) * gg
        v = b2 * v + (1 - b2) * gg * gg
        mh = m / (1 - b1 ** t); vh = v / (1 - b2 ** t)
        q = np.clip(q - rho * mh / (np.sqrt(vh) + eps), -3, 3); adm.append(q.copy())
    adm = np.array(adm)
    zp = np.array([loss(x, y) + 0.03 for x, y in plain])
    za = np.array([loss(x, y) + 0.03 for x, y in adm])
    for k in range(n_steps + 1):
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=DESC_AZIM)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=55,
                   zorder=7, label="minimum")
        ax.plot(plain[:k + 1, 0], plain[:k + 1, 1], zp[:k + 1], "-o",
                color="black", ms=3, lw=1.6, zorder=8, label="plain GD")
        ax.plot(adm[:k + 1, 0], adm[:k + 1, 1], za[:k + 1], "-o",
                color=ORANGE, ms=3.5, lw=2.0, zorder=9, label="Adam")
        ax.scatter([start[0]], [start[1]], [za[0]], color="magenta", s=50, zorder=10)
        ax.set_title(rf"Adam vs plain GD    step ${k}$", fontsize=11)
        ax.legend(loc="upper right", fontsize=8)
        if k % 5 == 0 or k < 5:
            fig.savefig(os.path.join(out, f"step_{k:02d}.pdf"), bbox_inches="tight")
        plt.close(fig)
    print("saved 41_adam: sampled frames of", n_steps + 1)

adam_frames()

saved 41_adam: sampled frames of 41
